# VideoFile

Represents a video file on disk and provides the core operations to create and process it.

---

## Construction

The class exposes two factory methods as alternatives to `__init__`:

- **`from_video(path)`** — loads an existing video from its path. Automatically reads metadata via OpenCV (dimensions, fps, frame count, duration).
- **`from_frames_dir(path, fps, crf, frames_dir)`** — creates a new video from a `FramesDir` instance, delegating encoding to FFmpeg with the given `fps` and `crf` (quality) parameters. Once the file is created, it behaves like a regular `VideoFile`.

---

## Attributes

After construction the object exposes:

| Attribute | Type | Description |
|---|---|---|
| `path` | `str` | Path to the video file |
| `width` | `int` | Width in pixels |
| `height` | `int` | Height in pixels |
| `fps` | `float` | Frames per second |
| `frame_count` | `int` | Total number of frames |
| `duration_sec` | `float` | Duration in seconds |

---

## Operations

- **`extract_frames(output_path)`** — extracts the video frames into a destination directory, delegating the operation to FFmpeg asynchronously.

---

## Dependencies

| Class | Role |
|---|---|
| `OpenCvRunner` | Reads video metadata |
| `FFmpegRunner` | Frame extraction and video creation |
| `FramesDir` | Frame source in `from_frames_dir` |

In [1]:
import movixpy.io as mvi

video_file = mvi.VideoFile(r".\data\01.mp4")
print(video_file.fps)
print(video_file.frame_count)
print(video_file.duration_sec)
print(video_file.width)
print(video_file.height)

output_dir = r".\data\01"
await video_file.extract_frames(
    r".\data\01"
)

25.0
742
29.68
1000
1000


# FramesDir

Represents a directory of image frames on disk and provides utilities to read, validate, and transform them into other media formats such as videos or GIFs.

---

## Construction

The class exposes three factory methods as alternatives to `__init__`:

* **`from_frames_dir(path)`** — loads an existing directory containing image frames. Only files with valid image extensions are considered and automatically sorted. Raises an error if the directory is invalid or contains no valid frames.

* **`from_image(path, image_path)`** — creates a frames directory starting from a single image. The image is copied into the target directory, which is created if it does not exist, and then treated as a one-frame sequence.

* **`from_video(path, video_file)`** — creates a frames directory by extracting frames from a `VideoFile` instance. The extraction is performed asynchronously via FFmpeg, and the resulting directory is then loaded as a standard `FramesDir`.

---

## Attributes

After construction the object exposes:

| Attribute | Type        | Description                    |
| --------- | ----------- | ------------------------------ |
| `path`    | `str`       | Path to the frames directory   |
| `frames`  | `list[str]` | Sorted list of frame filenames |
| `width`   | `int`       | Width of the frames in pixels  |
| `height`  | `int`       | Height of the frames in pixels |

---

## Behavior

The class behaves like a sequence:

* `len(frames_dir)` returns the number of frames.
* `frames_dir[i]` returns the full path of the i-th frame.

Frame dimensions are inferred from the first frame in the sequence via OpenCV.

---

## Operations

* **`create_video(output_path, fps, crf)`** — encodes the frames into a video file using FFmpeg. The output format must be one of the supported video extensions (`.mp4`, `.mov`, `.avi`). Encoding is performed asynchronously.

* **`create_gif(output_path)`** — encodes the frames into a GIF animation using FFmpeg. The output must have `.gif` extension.

* **`loop(target_frames)`** — extends the sequence to `target_frames` by cycling back to the beginning once the end is reached. Example: `[0,1,2,3] → [0,1,2,3,0,1,2,3,...]`. New frames are physically copied to disk and appended to `frames`.

* **`freeze(target_frames)`** — extends the sequence to `target_frames` by repeating the last frame indefinitely. Example: `[0,1,2,3] → [0,1,2,3,3,3,3,...]`. Useful to hold the final state of a clip.

* **`bounce(target_frames)`** — extends the sequence to `target_frames` with a ping-pong effect, reversing direction at each end. Example: `[0,1,2,3] → [0,1,2,3,2,1,0,1,2,3,...]`.

All three extension methods raise a `ValueError` if `target_frames` is not greater than the current frame count, and physically write the new frames to disk before updating the internal `frames` list.

---

## Validation Rules

* Only files with supported image extensions (`.jpg`, `.jpeg`, `.png`, `.bmp`, `.tiff`) are considered valid frames.
* The directory must contain at least one valid frame.
* Input types are strictly checked (e.g., `VideoFile` for video-based construction).
* Output formats are validated before encoding.
* Extension methods (`loop`, `freeze`, `bounce`) require `target_frames` to be strictly greater than the current frame count.

---

## Dependencies

| Class          | Role                                        |
| -------------- | ------------------------------------------- |
| `OpenCvRunner` | Reads frame dimensions                      |
| `FFmpegRunner` | Frame extraction and encoding (video/GIF)   |
| `VideoFile`    | Source for frame extraction in `from_video` |

---

## Design Insight

`FramesDir` and `VideoFile` form a bidirectional abstraction:

* `VideoFile → FramesDir` via frame extraction
* `FramesDir → VideoFile` via video encoding

This symmetry makes the API composable and allows building higher-level video processing pipelines in a clean and predictable way.

In [ ]:
frames_dir = mvi.FramesDir.from_frames_dir(output_dir)
# or mvi.FramesDir.from_video(output_dir, video_file), so create in automatically the frame dir extracting frames
print(len(frames_dir))
print(frames_dir[0])
print(frames_dir.width)
print(frames_dir.height)
frames_dir.bounce(target_frames=len(frames_dir) * 2)
await frames_dir.create_video(
    r".\data\01_from_frames.mp4",
    video_file.fps,
    23
)

742
.\data\01\00001.jpg
1000
1000


# MotionFramesDir

Represents a directory of image frames enriched with motion data, allowing each frame to be positioned, scaled, and rotated over time through keyframe-based interpolation.

It extends `FramesDir` by associating a sequence of `KeyPosition` objects that define how frames evolve across time.

---

## Construction

The class exposes three factory methods, mirroring `FramesDir` while adding motion support:

* **`from_frames_dir(path, key_positions)`** — loads an existing frames directory and associates it with a list of `KeyPosition`. The keyframes are validated and sorted by `frame_idx`.

* **`from_image(path, image_path, key_positions)`** — creates a single-frame sequence from an image and attaches motion data. Useful for animating static assets.

* **`from_video(path, video_file, key_positions)`** — extracts frames from a `VideoFile` and associates motion keyframes. Frame extraction is asynchronous.

All constructors ensure that keyframe indices are valid with respect to the number of frames.

---

## Attributes

In addition to inherited attributes from `FramesDir`, the object exposes:

| Attribute       | Type                | Description                                          |
| --------------- | ------------------- | ---------------------------------------------------- |
| `key_positions` | `list[KeyPosition]` | Sorted list of keyframes describing motion over time |

Each `KeyPosition` defines:

* `frame_idx` — the frame index where the keyframe is defined
* `angle` — rotation angle
* `position` — tuple `(x, y, w, h)` representing placement and size

---

## Behavior

The class inherits sequence behavior from `FramesDir`:

* `len(motion_frames_dir)` returns the number of frames
* `motion_frames_dir[i]` returns the full path of the i-th frame

Additionally, it introduces motion-aware operations based on interpolation between keyframes.

---

## Operations

* **`add(key_position)`** — adds a new keyframe. Existing keyframes within a small neighborhood of the same frame index are removed to avoid conflicts, then the list is re-sorted.

* **`remove(frame_idx)`** — removes the keyframe at the given frame index. Raises an error if no matching keyframe exists.

* **`get_frame_position(frame_idx)`** — computes the interpolated transformation `(angle, x, y, w, h)` for a given frame:

  * If the frame is before the first keyframe → returns the first keyframe
  * If after the last keyframe → returns the last keyframe
  * Otherwise → performs linear interpolation between the two nearest keyframes

Returns `None` if no keyframes are defined.

---

## Interpolation Model

Motion is computed using **linear interpolation** between consecutive keyframes:

* Rotation (`angle`) is interpolated linearly
* Position (`x`, `y`) is interpolated linearly (float precision)
* Size (`w`, `h`) is interpolated linearly and cast to integers

This produces smooth transitions across frames without requiring per-frame definitions.

---

## Validation Rules

* Every `KeyPosition.frame_idx` must be within `[0, frame_count - 1]`
* Keyframes are always stored sorted by frame index
* Invalid or out-of-range keyframes raise errors at construction time

---

## Dependencies

| Class          | Role                                  |
| -------------- | ------------------------------------- |
| `FramesDir`    | Base class providing frame management |
| `KeyPosition`  | Defines motion keyframes              |
| `VideoFile`    | Optional source for frame extraction  |
| `FFmpegRunner` | Used indirectly via `FramesDir`       |

---

## Design Insight

`MotionFramesDir` introduces a **timeline abstraction** on top of raw frames:

* `FramesDir` → static sequence of images
* `MotionFramesDir` → dynamic sequence with spatial-temporal behavior

This enables building higher-level systems such as:

* video compositing engines
* animated overlays
* programmatic motion graphics pipelines

The keyframe + interpolation model mirrors industry-standard animation systems, making the class both expressive and scalable for complex motion logic.


In [3]:
output_dir = r".\data\01_mov"
motion_frames_dir = await mvi.MotionFramesDir.from_video(
    output_dir,
    video_file,
    key_positions=[
        mvi.KeyPosition(0, (0, 0, 200, 200), angle=90),
        mvi.KeyPosition(video_file.frame_count - 1, (100, 100, 1000, 1000), angle=0)
    ]
)

print(motion_frames_dir.get_frame_position(10))
motion_frames_dir.add(key_position=mvi.KeyPosition(video_file.frame_count // 2, (50, 50, 500, 500), angle=45))
print(motion_frames_dir.get_frame_position(10))

(88, 1.349527665317139, 1.349527665317139, 210, 210)
(88, 1.3477088948787064, 1.3477088948787064, 208, 208)
